In [1]:
import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download

repo = "intronhealth/afrispeech-200"
csvs = [f for f in list_repo_files(repo, repo_type="dataset")
        if f.startswith("transcripts/") and f.endswith(".csv")]
print("Number of transcript files:", len(csvs))
print(csvs[:15])

c:\PROJECTS ML\nigerian-accented-clinical-asr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of transcript files: 227
['transcripts/afemai/test.csv', 'transcripts/afemai/train.csv', 'transcripts/afo/test.csv', 'transcripts/afrikaans/dev.csv', 'transcripts/afrikaans/test.csv', 'transcripts/afrikaans/train.csv', 'transcripts/agatu/test.csv', 'transcripts/akan-fante/dev.csv', 'transcripts/akan-fante/test.csv', 'transcripts/akan-fante/train.csv', 'transcripts/akan/test.csv', 'transcripts/akan/train.csv', 'transcripts/alago/test.csv', 'transcripts/alago/train.csv', 'transcripts/anaang/dev.csv']


In [2]:
print("Other files:", [f for f in csvs if f.count("/") != 2])

accents = ["yoruba", "igbo", "hausa"]
frames = []
for f in csvs:
    parts = f.split("/")
    if len(parts) == 3 and parts[1] in accents:
        df = pd.read_csv(hf_hub_download(repo, f, repo_type="dataset"))
        df["accent_folder"] = parts[1]
        df["split"] = parts[2].replace(".csv", "")
        frames.append(df)

meta = pd.concat(frames, ignore_index=True)
print("\nColumns:", meta.columns.tolist())

if {"duration", "domain"} <= set(meta.columns):
    summary = (meta.groupby(["accent_folder", "split", "domain"])
                   .agg(clips=("duration", "size"),
                        hours=("duration", lambda d: round(d.sum() / 3600, 1)))
                   .reset_index())
    print(summary.to_string(index=False))

Other files: ['transcripts/dev.csv', 'transcripts/test.csv', 'transcripts/train.csv']


c:\PROJECTS ML\nigerian-accented-clinical-asr\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chidi\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\PROJECTS ML\nigerian-accented-clinical-asr\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `hugg


Columns: ['idx', 'user_ids', 'accent', 'age_group', 'country', 'transcript', 'nchars', 'audio_ids', 'audio_paths', 'duration', 'origin', 'domain', 'split', 'gender', 'accent_folder']
accent_folder split   domain  clips  hours
        hausa   dev clinical     85    0.2
        hausa   dev  general     31    0.1
        hausa  test clinical    144    0.5
        hausa  test  general     52    0.1
        hausa train clinical   4602   16.0
        hausa train  general    835    2.8
         igbo   dev clinical    116    0.3
         igbo   dev  general    100    0.3
         igbo  test clinical    193    0.5
         igbo  test  general    162    0.4
         igbo train clinical   5521   17.1
         igbo train  general   2562    7.2
       yoruba   dev clinical    202    0.5
       yoruba   dev  general    159    0.4
       yoruba  test clinical    269    0.7
       yoruba  test  general    379    1.1
       yoruba train clinical   8864   26.3
       yoruba train  general   5505   15.8

In [3]:
train = meta[meta["split"] == "train"]
test = meta[meta["split"] == "test"]

overlap = set(train["user_ids"]) & set(test["user_ids"])
print("Speakers in both train and test:", len(overlap))

print(meta["duration"].describe().round(1))
print("Clips longer than 30s:", (meta["duration"] > 30).sum())

Speakers in both train and test: 0
count    29781.0
mean        10.9
std         10.2
min          1.0
25%          5.3
50%          8.7
75%         13.8
max        345.1
Name: duration, dtype: float64
Clips longer than 30s: 769


In [4]:
import os
os.makedirs("../data", exist_ok=True)

ok = meta[meta["duration"] <= 30]
print("Test clips removed (over 30s):",
      ((meta["split"] == "test") & (meta["duration"] > 30)).sum())

train_parts = []
for acc in accents:
    d = ok[(ok["split"] == "train") & (ok["accent_folder"] == acc)]
    d = d.sample(frac=1, random_state=42)         
    d = d[d["duration"].cumsum() <= 5 * 3600]     
    train_parts.append(d)

subsets = {
    "train": pd.concat(train_parts),
    "dev": ok[ok["split"] == "dev"],
    "test": ok[ok["split"] == "test"],
}

for name, d in subsets.items():
    d.to_csv(f"../data/{name}_subset.csv", index=False)
    print(f"{name}: {len(d)} clips, {d['duration'].sum()/3600:.1f} hours")

print(subsets["train"].groupby(["accent_folder", "domain"]).size())

Test clips removed (over 30s): 14
train: 5431 clips, 15.0 hours
dev: 689 clips, 1.7 hours
test: 1185 clips, 3.0 hours
accent_folder  domain  
hausa          clinical    1463
               general      292
igbo           clinical    1245
               general      604
yoruba         clinical    1101
               general      726
dtype: int64
